# CineEmbed — EDA v2

Pipeline-first refresh of the EDA notebook. Applies all 13 fixes from the audit (see `docs/superpowers/specs/2026-05-03-eda-v2-design.md`). Produces a clean (329044, 451) feature matrix.

**Sections:**
- §1 Setup & Reproducibility
- §2 Pipeline Function Definitions
- §3 Pipeline Execution
- §4 EDA Visualizations
- §5 Persistence


In [ ]:
# §1 — Setup & Reproducibility
import os, json, hashlib, random, warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, MultiLabelBinarizer
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
from scipy import stats

# Optional GPU stack — only imported if available
try:
    import torch
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    if HAS_TORCH:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


CONFIG = {
    'seed': 42,
    'data_dir': Path('data'),
    'artifacts_dir': Path('artifacts'),
    'figures_dir': Path('artifacts/figures'),

    # File paths
    'paths': {
        'details': Path('data/AllMoviesDetailsCleaned.csv'),
        'casting': Path('data/AllMoviesCastingRaw.csv'),
        'awards':  Path('data/220k_awards_by_directors.csv'),
    },

    # Feature engineering knobs (single source of truth — fix #11 + clean ablation)
    'top_n_genres': 20,
    'top_n_languages': 30,
    'q99_clip_threshold': 0.99,
    'runtime_clip': (10, 300),

    # Embedding model (fix #5 — multilingual)
    'embedding_model': 'paraphrase-multilingual-MiniLM-L12-v2',
    'embedding_batch_size': 64,
    'embedding_dim': 384,
    'embedding_cache': Path('artifacts/text_embeddings.npy'),
    'embedding_meta': Path('artifacts/text_embeddings.meta.json'),
}

CONFIG['artifacts_dir'].mkdir(parents=True, exist_ok=True)
CONFIG['figures_dir'].mkdir(parents=True, exist_ok=True)

seed_everything(CONFIG['seed'])

# Reproducibility self-check
_check = np.random.rand(3)
print("\u2705 \u00a71 Setup complete")
print(f"   seed = {CONFIG['seed']}")
print(f"   np.random sample (deterministic) = {_check}")
print(f"   torch available = {HAS_TORCH}")


## §2.1 — Data Layer
Pure functions: `load_csvs`, `normalize_director_name`, `merge_details_casting`. Each function has a sanity-test cell immediately after.

In [ ]:
# §2.1 — Data layer functions
import unicodedata

def normalize_director_name(name: str | None) -> str:
    """Stable director key for joins.

    Steps:
      1. None / NaN / empty → ''
      2. Unicode NFKD decompose, strip combining marks (accents)
      3. Swap "Last, First" → "First Last"
      4. Lowercase, collapse internal whitespace, strip ends.
    """
    if name is None or (isinstance(name, float) and np.isnan(name)):
        return ''
    s = str(name).strip()
    if not s:
        return ''
    # NFKD + ascii filter (drops accents)
    s = unicodedata.normalize('NFKD', s)
    s = ''.join(ch for ch in s if not unicodedata.combining(ch))
    # "Last, First" → "First Last"
    if ',' in s:
        parts = [p.strip() for p in s.split(',', 1)]
        if len(parts) == 2 and parts[0] and parts[1]:
            s = f"{parts[1]} {parts[0]}"
    # collapse whitespace, lowercase
    s = ' '.join(s.split()).lower()
    return s


In [ ]:
# Test §2.1: normalize_director_name (fix #8)
_cases = [
    ('Steven Spielberg',   'steven spielberg'),
    ('Spielberg, Steven',  'steven spielberg'),
    ('  Pedro  Almodóvar ', 'pedro almodovar'),
    ('Léa  Pool',          'lea pool'),
    (None,                 ''),
    ('',                   ''),
    ('SCORSESE, MARTIN',   'martin scorsese'),
]
for raw, expected in _cases:
    got = normalize_director_name(raw)
    assert got == expected, f"normalize_director_name({raw!r}) = {got!r} != {expected!r}"
print(f"\u2705 normalize_director_name: {len(_cases)} cases pass")


In [ ]:
def load_csvs(paths: dict[str, Path]) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Load the three source CSVs with their correct separators.

    details, casting use ';' (TMDB-derived); awards uses ','.
    """
    details = pd.read_csv(paths['details'], sep=';', low_memory=False)
    casting = pd.read_csv(paths['casting'], sep=';', low_memory=False)
    awards  = pd.read_csv(paths['awards'],  sep=',', low_memory=False)
    return details, casting, awards


In [ ]:
def merge_details_casting(details: pd.DataFrame, casting: pd.DataFrame) -> pd.DataFrame:
    """Inner-shape merge on `id`. Adds `director_name_norm` (fix #8) for award joins.

    Carries `director_name` (raw) for human readability and `director_name_norm`
    for joins. Only the necessary casting columns are pulled in to avoid bloat.
    """
    casting_slim = casting[['id', 'director_name', 'director_gender']].copy()
    casting_slim['director_name_norm'] = casting_slim['director_name'].apply(normalize_director_name)
    merged = details.merge(casting_slim, on='id', how='left')
    return merged


In [ ]:
# Test §2.1: merge_details_casting
_details = pd.DataFrame({'id': [1, 2, 3], 'title': ['A', 'B', 'C']})
_casting = pd.DataFrame({
    'id': [1, 2, 3],
    'director_name': ['Steven Spielberg', 'Spielberg, Steven', None],
    'director_gender': [2, 2, 0],
})
_merged = merge_details_casting(_details, _casting)
assert set(_merged.columns) >= {'id', 'title', 'director_name', 'director_name_norm', 'director_gender'}
assert _merged.loc[0, 'director_name_norm'] == 'steven spielberg'
assert _merged.loc[1, 'director_name_norm'] == 'steven spielberg'   # normalized form merges
assert _merged.loc[2, 'director_name_norm'] == ''
print("✅ merge_details_casting passes")


## §2.2 — Awards Layer
Temporal-aware per-film aggregation. Implements fixes #1 (temporal leak) and #9 (Oscar/Palme regex word-boundary).

In [ ]:
import re

# OSCAR_RE: Word-boundary match for "Oscar" with a negative lookahead that
# excludes "Oscar Wilde Award" (Oscar Wilde is a person, not the Oscar award).
# The plan's audit identified this as a real-world false-positive trap.
OSCAR_RE = re.compile(r'\bOscar\b(?!\s+Wilde)', flags=re.IGNORECASE)
PALME_RE = re.compile(r'\bPalme\b', flags=re.IGNORECASE)


def aggregate_awards_temporal(
    awards: pd.DataFrame,
    films: pd.DataFrame,
) -> pd.DataFrame:
    """Per-film aggregation of director awards.

    Two fixes:
      - #1 (temporal): only awards with year <= film.release_year contribute.
      - #9 (regex):    Oscar/Palme detection uses \\b word-boundary.

    Returns a DataFrame with one row per film id and columns:
      prior_total_nominations, prior_total_wins,
      prior_oscar_nominations, prior_oscar_wins,
      prior_palme_nominations, prior_palme_wins

    Note: *_nominations counts entries where the director was nominated but did
    NOT win (i.e., is_oscar/is_palme AND NOT is_won). *_wins counts entries
    where they won (is_oscar/is_palme AND is_won).
    """
    # Normalize director key on awards side
    awards = awards.copy()
    awards['director_name_norm'] = awards['director_name'].apply(normalize_director_name)

    # Year column (if missing, parse from a date-like field; here we trust 'year')
    awards['year'] = pd.to_numeric(awards['year'], errors='coerce')
    awards = awards.dropna(subset=['year'])
    awards['year'] = awards['year'].astype(int)

    # Annotate flags up-front (avoid per-row regex during the join)
    awards['is_won']   = (awards['outcome'].fillna('') == 'Won')
    awards['is_oscar'] = awards['category'].fillna('').str.contains(OSCAR_RE, regex=True)
    awards['is_palme'] = awards['category'].fillna('').str.contains(PALME_RE, regex=True)

    # Films side — derive year
    films = films.copy()
    films['release_year'] = pd.to_datetime(films['release_date'], errors='coerce').dt.year

    # For each (director_norm), pre-sort awards ascending by year for cumulative aggregation
    award_groups = awards.sort_values('year').groupby('director_name_norm')

    rows = []
    for _, film in films[['id', 'director_name_norm', 'release_year']].iterrows():
        out = {
            'id': film['id'],
            'prior_total_nominations': 0,
            'prior_total_wins': 0,
            'prior_oscar_nominations': 0,
            'prior_oscar_wins': 0,
            'prior_palme_nominations': 0,
            'prior_palme_wins': 0,
        }
        director = film['director_name_norm']
        year = film['release_year']
        if not director or pd.isna(year):
            rows.append(out)
            continue
        if director not in award_groups.groups:
            rows.append(out)
            continue
        sub = award_groups.get_group(director)
        sub = sub[sub['year'] <= year]
        if sub.empty:
            rows.append(out)
            continue
        out['prior_total_nominations']  = int((~sub['is_won']).sum())
        out['prior_total_wins']         = int(sub['is_won'].sum())
        out['prior_oscar_nominations']  = int((sub['is_oscar'] & ~sub['is_won']).sum())
        out['prior_oscar_wins']         = int((sub['is_oscar'] &  sub['is_won']).sum())
        out['prior_palme_nominations']  = int((sub['is_palme'] & ~sub['is_won']).sum())
        out['prior_palme_wins']         = int((sub['is_palme'] &  sub['is_won']).sum())
        rows.append(out)

    return pd.DataFrame(rows)


In [ ]:
# Test §2.2: aggregate_awards_temporal
# Synthetic awards: director "alice" has 2 wins (1995, 2010), 1 nom (2005)
# Director "bob" has 1 oscar win (2000) and 1 "Oscar Wilde Award" nom (1998 — false positive trap)
_awards = pd.DataFrame({
    'director_name': ['Alice', 'Alice', 'Alice', 'Bob', 'Bob'],
    'category':      ['Best Director', 'Best Picture', 'Best Director — Oscar',
                      'Academy Award (Oscar)', 'Oscar Wilde Award'],
    'outcome':       ['Won', 'Nominated', 'Won', 'Won', 'Nominated'],
    'year':          [1995, 2005, 2010, 2000, 1998],
})
_films = pd.DataFrame({
    'id': [101, 102, 103, 104],
    'director_name_norm': ['alice', 'alice', 'bob', 'bob'],
    'release_date': pd.to_datetime(['1990-01-01', '2008-01-01', '1995-01-01', '2005-01-01']),
})
_agg = aggregate_awards_temporal(_awards, _films)

# Assertions (one per fix):
# Fix #1 — temporal: alice 1990 film sees 0 wins; alice 2008 film sees 1 win (1995)
row_a1990 = _agg.set_index('id').loc[101]
row_a2008 = _agg.set_index('id').loc[102]
assert row_a1990['prior_total_wins'] == 0,  f"1990 leak: {row_a1990['prior_total_wins']}"
assert row_a2008['prior_total_wins'] == 1,  f"2008 expects 1 win, got {row_a2008['prior_total_wins']}"

# Fix #9 — word-boundary: bob 1995 sees 0 oscar wins (Wilde shouldn't count, win is 2000)
# bob 2005 sees 1 oscar win (2000) and 0 oscar noms (Wilde is filtered out)
row_b1995 = _agg.set_index('id').loc[103]
row_b2005 = _agg.set_index('id').loc[104]
assert row_b1995['prior_oscar_wins'] == 0
assert row_b2005['prior_oscar_wins'] == 1
assert row_b2005['prior_oscar_nominations'] == 0, \
    f"Wilde should not count, got {row_b2005['prior_oscar_nominations']}"

print("\u2705 aggregate_awards_temporal: temporal cutoff + regex word-boundary work")


## §2.3 — Feature Engineering
Five engineer_* functions, one per modality. Each returns a DataFrame block aligned by row index with the master frame.

### §2.3.a — Numerical (fixes #2, #7, #10)

In [ ]:
def engineer_numerical(df: pd.DataFrame, config: dict) -> pd.DataFrame:
    """Numerical block — 6 columns.

    Fixes:
      #2 vote_average imputation — use mean of voted-only films, NOT median (which is 0)
      #7 vote_count Q99 clip + log (was: log only)
      #10 has_engagement flag
    """
    out = pd.DataFrame(index=df.index)

    pop = pd.to_numeric(df['popularity'], errors='coerce').fillna(0)
    vc  = pd.to_numeric(df['vote_count'], errors='coerce').fillna(0)
    rt  = pd.to_numeric(df['runtime'], errors='coerce')
    va  = pd.to_numeric(df['vote_average'], errors='coerce')

    # Q99 clip then log1p
    pop_q99 = pop.quantile(config['q99_clip_threshold'])
    vc_q99  = vc.quantile(config['q99_clip_threshold'])
    out['log_popularity'] = np.log1p(pop.clip(upper=pop_q99).clip(lower=0))
    out['log_vote_count'] = np.log1p(vc.clip(upper=vc_q99).clip(lower=0))

    # runtime: cap to [10, 300] then min-max
    rt_lo, rt_hi = config['runtime_clip']
    rt_filled = rt.fillna(rt.median() if rt.notna().any() else (rt_lo + rt_hi) / 2)
    rt_capped = rt_filled.clip(lower=rt_lo, upper=rt_hi)
    out['runtime_norm'] = (rt_capped - rt_lo) / (rt_hi - rt_lo)

    # vote_average — fix #2 (smart imputation)
    voted_mask = vc > 0
    if voted_mask.any():
        imputed_value = va[voted_mask].mean()  # ~6.0 on real data
    else:
        imputed_value = 0.0
    va_filled = va.where(voted_mask & va.notna(), imputed_value)
    # min-max on [0, 10] natural scale
    out['vote_average_norm'] = (va_filled.clip(0, 10) / 10.0)

    # Flags — fixes #2, #10
    out['has_vote']        = voted_mask.astype(np.int8)
    out['has_engagement']  = ((pop > 0) | (vc > 0)).astype(np.int8)

    return out

In [ ]:
# Test §2.3.a: engineer_numerical
_df = pd.DataFrame({
    'popularity':   [0.0, 0.5, 1.0, 100.0, np.nan, 0.0],
    'vote_count':   [0,   10,  50,  10000, 5,      0],
    'runtime':      [120, 95,  np.nan, 5,   500,   90],
    'vote_average': [np.nan, 7.5, 8.0, 6.5, 5.0, np.nan],
})
_out = engineer_numerical(_df, CONFIG)

# fix #2: vote_average imputation
# Voted-only mean = mean of [7.5, 8.0, 6.5, 5.0] = 6.75
# Films with vote_count == 0 (rows 0 and 5) → vote_average filled with ~6.75
assert abs(_out.loc[0, 'vote_average_norm'] - _out.loc[5, 'vote_average_norm']) < 1e-9
# Both should NOT be the minimum (which would happen with median=0 imputation)
assert _out['vote_average_norm'].min() < _out.loc[0, 'vote_average_norm'] or \
       _out['vote_average_norm'].max() > _out.loc[0, 'vote_average_norm']

# fix #7: vote_count Q99 clip applied — log_vote_count for 10000 should not blow up
assert _out['log_vote_count'].max() < np.log1p(10001)

# fix #10: has_engagement flag
assert _out.loc[0, 'has_engagement'] == 0  # popularity=0 AND vote_count=0
assert _out.loc[1, 'has_engagement'] == 1  # vote_count=10 > 0
assert _out.loc[5, 'has_engagement'] == 0

# has_vote flag
assert _out.loc[0, 'has_vote'] == 0
assert _out.loc[1, 'has_vote'] == 1

# All 6 columns present, no NaN
expected_cols = {'log_popularity', 'log_vote_count', 'runtime_norm',
                 'vote_average_norm', 'has_vote', 'has_engagement'}
assert set(_out.columns) == expected_cols
assert _out.isna().sum().sum() == 0

print(f"✅ engineer_numerical: 6-col output, {len(_out)} rows, no NaN")

### §2.3.b — Genre & Language (fix #3 — Unknown genre + has_genre flag)

In [ ]:
def engineer_genres(df: pd.DataFrame, top_n: int = 20) -> pd.DataFrame:
    """Genre block — top-N multi-hot + Unknown + has_genre flag (fix #3).

    Empty/None genre lists become ['Unknown'] AND has_genre = 0.
    """
    raw = df['genres'].fillna('').astype(str)
    genres_list = raw.apply(
        lambda x: [g.strip() for g in x.split('|') if g.strip()]
    )
    has_genre = genres_list.apply(lambda lst: 1 if lst else 0).astype(np.int8)

    # Pick top-N from non-empty rows
    counts = {}
    for lst in genres_list:
        for g in lst:
            counts[g] = counts.get(g, 0) + 1
    top_genres = [g for g, _ in sorted(
        counts.items(),
        key=lambda kv: (-kv[1], kv[0]),
    )[:top_n]]

    # Replace empty with ['Unknown'] for encoding
    genres_for_mlb = genres_list.apply(lambda lst: lst if lst else ['Unknown'])

    # Restrict to top_genres + 'Unknown'
    keep = set(top_genres) | {'Unknown'}
    genres_filtered = genres_for_mlb.apply(lambda lst: [g for g in lst if g in keep] or ['Unknown'])

    mlb = MultiLabelBinarizer(classes=sorted(keep))
    encoded = mlb.fit_transform(genres_filtered)
    out = pd.DataFrame(
        encoded,
        columns=[f'genre_{g}' for g in mlb.classes_],
        index=df.index,
        dtype=np.int8,
    )
    out['has_genre'] = has_genre.values
    return out

In [ ]:
# Test §2.3.b: engineer_genres (fix #3)
_df = pd.DataFrame({
    'genres': ['Drama|Comedy', 'Drama', None, '', 'Action|Thriller', 'Drama|Comedy'],
})
_out = engineer_genres(_df, top_n=3)

# Top 3 by frequency: Drama (3), Comedy (2), Action (1) (or Thriller — tiebreak by alpha)
# So expected columns: genre_Drama, genre_Comedy, genre_Action OR Thriller, genre_Unknown, has_genre
expected_genre_cols = {'genre_Drama', 'genre_Comedy', 'genre_Unknown', 'has_genre'}
assert expected_genre_cols.issubset(set(_out.columns)), f"missing cols. got: {set(_out.columns)}"

# Fix #3: empty/None → genre_Unknown=1, has_genre=0
assert _out.loc[2, 'genre_Unknown'] == 1
assert _out.loc[3, 'genre_Unknown'] == 1
assert _out.loc[2, 'has_genre'] == 0
assert _out.loc[3, 'has_genre'] == 0

# Non-empty rows: has_genre = 1, genre_Unknown = 0
assert _out.loc[0, 'has_genre'] == 1
assert _out.loc[0, 'genre_Unknown'] == 0
assert _out.loc[0, 'genre_Drama'] == 1
assert _out.loc[0, 'genre_Comedy'] == 1

print(f"✅ engineer_genres: {len(_out.columns)} cols, fix #3 holds")

### §2.3.c — Language

In [ ]:
def engineer_languages(df: pd.DataFrame, top_n: int = 30) -> pd.DataFrame:
    """Language block — top-N + 'other' one-hot."""
    raw = df['original_language'].fillna('other').astype(str)
    counts = raw.value_counts()
    top_langs = counts.head(top_n).index.tolist()
    grouped = raw.where(raw.isin(top_langs), other='other')
    out = pd.get_dummies(grouped, prefix='lang', dtype=np.int8)
    # Ensure 'lang_other' column exists even if no rows fell out of top-N
    if 'lang_other' not in out.columns:
        out['lang_other'] = np.int8(0)
    return out

In [ ]:
# Test §2.3.b: engineer_languages
_df = pd.DataFrame({
    'original_language': ['en', 'en', 'fr', 'tr', 'jp', 'unknown_lang_xyz', None],
})
_out = engineer_languages(_df, top_n=3)
# Top 3: en (2), fr (1), jp/tr (1) — alpha tiebreak
expected_subset = {'lang_en', 'lang_fr', 'lang_other'}
assert expected_subset.issubset(set(_out.columns))
assert _out.loc[5, 'lang_other'] == 1   # unknown_lang_xyz
assert _out.loc[6, 'lang_other'] == 1   # None

# Each row has exactly one '1' across all lang columns (one-hot)
assert (_out.sum(axis=1) == 1).all()
print(f"✅ engineer_languages: one-hot, {len(_out.columns)} cols")

### §2.3.d — Decade (fix #6)

In [ ]:
def engineer_decade(df: pd.DataFrame) -> pd.DataFrame:
    """Decade block — fix #6.

    For films with parseable release_date:
      decade_norm = (decade - 1900) / 130, where decade = year // 10 * 10.
    For films without:
      decade_norm = 0.0 AND has_release_date = 0 — model can learn to ignore decade.
    """
    rd = pd.to_datetime(df['release_date'], errors='coerce')
    has = rd.notna()
    year = rd.dt.year.where(has, other=np.nan)
    decade = (year // 10 * 10).where(has, other=np.nan)
    decade_norm = ((decade - 1900) / 130).where(has, other=0.0)

    out = pd.DataFrame({
        'decade_norm': decade_norm.astype(float).values,
        'has_release_date': has.astype(np.int8).values,
    }, index=df.index)
    return out

In [ ]:
# Test §2.3.c: engineer_decade (fix #6)
_df = pd.DataFrame({
    'release_date': ['1990-05-01', '2020-01-01', '1900-01-01', None, 'invalid'],
})
_df['release_date'] = pd.to_datetime(_df['release_date'], errors='coerce')
_out = engineer_decade(_df)

assert set(_out.columns) == {'decade_norm', 'has_release_date'}
# (1990 - 1900) / 130 = 0.6923...
assert abs(_out.loc[0, 'decade_norm'] - (90/130)) < 1e-9
# (2020 - 1900) / 130 = 0.923...
assert abs(_out.loc[1, 'decade_norm'] - (120/130)) < 1e-9
# 1900 → 0.0
assert _out.loc[2, 'decade_norm'] == 0.0
# Missing/invalid → decade_norm = 0.0 AND has_release_date = 0
assert _out.loc[3, 'decade_norm'] == 0.0
assert _out.loc[3, 'has_release_date'] == 0
assert _out.loc[4, 'decade_norm'] == 0.0
assert _out.loc[4, 'has_release_date'] == 0
# Valid rows → has_release_date = 1
assert _out.loc[0, 'has_release_date'] == 1
assert _out.loc[2, 'has_release_date'] == 1

print("✅ engineer_decade: fix #6 (normalized + has_release_date) holds")